# IRIS-X ARC-AGI-2 prompt-pilot baseline 002
Private, Internet-disabled, competition-mounted L4 x4 replacement authorized by Decision 0017. This notebook resolves only identity-specific modern or legacy Kaggle mounts, then runs exactly the frozen 384 baseline jobs with one persistent model worker per GPU, no warm-up generation, retry, reflection, or competition submission.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import stat
import subprocess
import sys
import zipfile

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['PYTHONUNBUFFERED'] = '1'

def resolve_exact_directory(label, candidates):
    matches = [path for path in candidates if path.is_dir()]
    if len(matches) != 1:
        raise RuntimeError(f'{label} mount must resolve exactly once; candidates={candidates}, matches={matches}')
    return matches[0]

def resolve_exact_file(label, candidates):
    matches = [path for path in candidates if path.is_file()]
    if len(matches) != 1:
        raise RuntimeError(f'{label} file must resolve exactly once; candidates={candidates}, matches={matches}')
    return matches[0]

base_root = resolve_exact_directory('base Dataset', [
    Path('/kaggle/input/datasets/danielwuu/irisx-arc-agi-2-trial-002-smoke-001'),
    Path('/kaggle/input/irisx-arc-agi-2-trial-002-smoke-001'),
])
runtime_root = resolve_exact_directory('runtime Dataset', [
    Path('/kaggle/input/datasets/danielwuu/irisx-arc-agi-2-prompt-pilot-baseline-001'),
    Path('/kaggle/input/irisx-arc-agi-2-prompt-pilot-baseline-001'),
])
competition_challenges = resolve_exact_file('competition challenge', [
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json'),
    Path('/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json'),
])
archive_candidate = runtime_root / 'prompt-pilot-baseline-002.zip'
extracted_candidate = runtime_root / 'prompt-pilot-baseline-002'
archive_present = archive_candidate.is_file()
extracted_present = (extracted_candidate / 'manifest.json').is_file()
if int(archive_present) + int(extracted_present) != 1:
    raise RuntimeError('exact baseline-002 outbound must have exactly one archive or extracted representation')
if archive_present:
    outbound = archive_candidate
    outbound_form = 'archive'
else:
    outbound = Path('/kaggle/working/prompt-pilot-baseline-002.zip')
    outbound_form = 'exact-extracted-members'
    members = [
        'manifest.json', 'bindings/run.json', 'cases/solver-cases.json',
        'runtime/core.py', 'runtime/dependencies.json', 'runtime/runner.py',
        'sources/base-wheel-manifest.json', 'sources/model-manifest.json',
        'sources/runtime-wheel-manifest.json',
    ]
    sources = [extracted_candidate / member for member in members]
    if not all(source.is_file() for source in sources):
        raise RuntimeError('identity-specific extracted outbound is missing an exact bound member')
    with zipfile.ZipFile(outbound, 'w', allowZip64=True) as archive:
        for member, source in zip(members, sources):
            info = zipfile.ZipInfo(member, (1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_STORED
            info.create_system = 3
            info.external_attr = (stat.S_IFREG | 0o644) << 16
            archive.writestr(info, source.read_bytes())
runtime_wheel = runtime_root / 'irisx-0.2.0-py3-none-any.whl'
model_root = base_root
EXPECTED_OUTBOUND_SHA256 = '__IRISX_OUTBOUND_SHA256__'
EXPECTED_RUNTIME_WHEEL_SHA256 = '__IRISX_RUNTIME_WHEEL_SHA256__'
required_files = [runtime_wheel, model_root / 'config.json', model_root / 'model.safetensors.index.json', base_root / 'transformers-5.8.1-py3-none-any.whl']
if not all(path.is_file() for path in required_files):
    raise RuntimeError('one or more explicitly bound Dataset files are missing')
if hashlib.sha256(outbound.read_bytes()).hexdigest() != EXPECTED_OUTBOUND_SHA256:
    raise RuntimeError('outbound digest differs from technical preflight')
if hashlib.sha256(runtime_wheel.read_bytes()).hexdigest() != EXPECTED_RUNTIME_WHEEL_SHA256:
    raise RuntimeError('runtime wheel digest differs from technical preflight')
mounts = {'run_id': 'prompt-pilot-baseline-002', 'base_root': str(base_root), 'runtime_root': str(runtime_root), 'competition_challenges': str(competition_challenges), 'outbound_form': outbound_form}
print(json.dumps({'event': 'mount_paths_resolved', **mounts}, sort_keys=True), flush=True)

import torch
hardware = []
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    hardware.append({'index': index, 'name': props.name, 'memory_mib': int(props.total_memory // (1024 * 1024)), 'uuid': str(getattr(props, 'uuid', 'unknown'))})
smi = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,uuid,pci.bus_id', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False, timeout=30)
preflight = {'run_id': 'prompt-pilot-baseline-002', 'torch_cuda_device_count': torch.cuda.device_count(), 'gpus': hardware, 'nvidia_smi_exit_code': smi.returncode, 'nvidia_smi_rows': [line.strip() for line in smi.stdout.splitlines() if line.strip()]}
Path('/kaggle/working/notebook-preflight.json').write_text(json.dumps(preflight, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({'event': 'notebook_hardware_preflight', **preflight}, sort_keys=True), flush=True)
if torch.cuda.device_count() != 4 or len(hardware) != 4 or len({item['uuid'] for item in hardware}) != 4 or any('L4' not in item['name'] or item['memory_mib'] < 22000 for item in hardware) or smi.returncode != 0 or len(preflight['nvidia_smi_rows']) != 4:
    raise RuntimeError(f'four-L4 notebook preflight failed: {preflight}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(base_root), '--upgrade', '--no-deps', 'transformers==5.8.1', 'jinja2==3.1.6'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--force-reinstall', '--no-deps', str(runtime_wheel)], check=True)
command = [sys.executable, '-u', '-m', 'irisx.kaggle_runner_prompt_pilot', '--outbound', str(outbound), '--model-root', str(model_root), '--competition-challenges', str(competition_challenges), '--output', '/kaggle/working/result.zip', '--checkpoint-output', '/kaggle/working/checkpoint.json', '--progress-output', '/kaggle/working/progress.jsonl']
subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
